1° PASSO- Carregar o arquivo CSV

In [1]:
import pandas as pd
import re
from datetime import datetime

In [2]:
CAMINHO = 'C:\\turma-visualizacao-de-dados\\alunos\\andressa_alves\\semana_04\\base_rh.csv'
df = pd.read_csv(CAMINHO, 
                 sep=';',
                 encoding='cp1252',
                 decimal=','
                 )
print('Dados carregados com sucesso! ')
print('Número de linhas e colunas:', df.shape)
print('Número de linhas: ', df.shape[0])
print('Número de colunas: ', df.shape[1])


Dados carregados com sucesso! 
Número de linhas e colunas: (1000, 10)
Número de linhas:  1000
Número de colunas:  10


2° PASSO- Diagnóstico

In [3]:
df.info()
df.describe().round(2)

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   ID_Funcionario  1000 non-null   int64  
 1   Nome            1000 non-null   str    
 2   Departamento    1000 non-null   str    
 3   Cargo           1000 non-null   str    
 4   Salario         1000 non-null   float64
 5   Data_Admissao   1000 non-null   str    
 6   Genero          1000 non-null   str    
 7   Idade           1000 non-null   int64  
 8   Estado_Civil    1000 non-null   str    
 9   Status          1000 non-null   str    
dtypes: float64(1), int64(2), str(7)
memory usage: 78.3 KB


,ID_Funcionario,Salario,Idade
count,1000.00,1000.00,1000.00
mean,500.50,8579.95,41.40
std,288.82,3657.37,13.67
min,1.00,2000.71,18.00
25%,250.75,5564.54,30.00
50%,500.50,8571.13,41.00
75%,750.25,11554.63,53.00
max,1000.00,14954.51,65.00


3° PASSO- Converter datas e criar colunas 

In [4]:
#Converter data
print(f'Tipo antes: {df["Data_Admissao"].dtype}')
print(f'Valor antes: {df["Data_Admissao"].iloc[0]}')
print()

df_limpo = df.copy()

df_limpo['Data_Admissao'] = pd.to_datetime(
    df_limpo['Data_Admissao'], 
    format='%d/%m/%Y', 
    errors='coerce')

print(f'Tipo depois: {df_limpo["Data_Admissao"].dtype}')
print(f'Valor depois: {df_limpo["Data_Admissao"].iloc[0]}')
print(f'NaT gerados: {df_limpo["Data_Admissao"].isna().sum()}')

# Criar colunas Ano_Admissao, Mes_Admissao, Tempo_casa_Anos e Faixa_Tempo_Casa

df_limpo['Ano_Admissão'] = df_limpo['Data_Admissao'].dt.year
df_limpo['Mês_Admissão'] = df_limpo['Data_Admissao'].dt.month
df_limpo['Trimestre'] = df_limpo['Data_Admissao'].dt.quarter

hoje_ts = pd.Timestamp.today()

df_limpo['Tempo_Casa_Anos'] = (
    (hoje_ts - df_limpo['Data_Admissao']).dt.days / 365.25
).round(1)

df_limpo['Faixa_Tempo_Casa'] = pd.cut(
    df_limpo['Tempo_Casa_Anos'],
    bins=[0, 1, 3, 5, float('inf')],
    labels=['Até 1 ano', 'De 1 a 3 anos', 'De 3 a 5 anos', 'Mais de 5 anos']
)

cools = ['Nome', 'Data_Admissao', 'Ano_Admissão', 'Mês_Admissão', 'Tempo_Casa_Anos', 'Faixa_Tempo_Casa']
df_limpo[cools].head(8)


Tipo antes: str
Valor antes: 13/08/2024

Tipo depois: datetime64[us]
Valor depois: 2024-08-13 00:00:00
NaT gerados: 0


,Nome,Data_Admissao,Ano_Admissão,Mês_Admissão,Tempo_Casa_Anos,Faixa_Tempo_Casa
0,Julia Nunes,2024-08-13,2024,8,1.7,De 1 a 3 anos
1,Sr. Gustavo Duarte,2017-04-29,2017,4,9.0,Mais de 5 anos
2,Srta. Mariana Cunha,2024-12-11,2024,12,1.4,De 1 a 3 anos
3,Ana Sophia da Cruz,2019-06-16,2019,6,6.9,Mais de 5 anos
4,Dr. Nicolas Pinto,2019-03-29,2019,3,7.1,Mais de 5 anos
5,Marcela Costa,2024-07-23,2024,7,1.8,De 1 a 3 anos
6,João Miguel da Costa,2021-11-11,2021,11,4.5,De 3 a 5 anos
7,Gabrielly Cardoso,2022-04-26,2022,4,4.0,De 3 a 5 anos


4° PASSO- groupby - Maior e Menor Média de tempo de casa

In [5]:
media_depto = (
    df_limpo.groupby('Departamento')['Tempo_Casa_Anos']
    .mean()
    .round(1)
    .sort_values(ascending=False)
    .rename('Media_Anos')
    .reset_index())

print('Tempo médio de casa por departamento: ')
print(media_depto.to_string(index=False))

maior = media_depto.set_index('Departamento')['Media_Anos'].idxmax()
menor = media_depto.set_index('Departamento')['Media_Anos'].idxmin()

print(f'\n -> Maior retenção: {maior}')
print(f' -> Menor retenção: {menor}')

Tempo médio de casa por departamento: 
Departamento  Media_Anos
          RH         5.8
      Vendas         5.7
    Produção         5.5
          TI         5.5
   Logística         5.3
  Financeiro         5.1

 -> Maior retenção: RH
 -> Menor retenção: Financeiro


5° PASSO - Exporte base_rh_deptos.xlsx com uma aba 'Completo' e mais uma aba por departamento.

In [6]:
df.to_excel('base_rh.xlsx', index=False, sheet_name='Funcionarios')
print('base_rh.xlsx (aba única) gerada!')

with pd.ExcelWriter('base_rh_por_departamento.xlsx', engine='openpyxl') as writer:
    df.to_excel(writer, index=False, sheet_name='Todos')
    print('Aba Todos:  todos os registros.')
   
    for depto in sorted(df['Departamento'].unique()):
        df_depto = df[df['Departamento'] == depto]

        nome_aba = depto[:31]  # Limita o nome da aba a 31 caracteres

        df_depto.to_excel(writer, index=False, sheet_name=nome_aba)
        print(f'Aba {nome_aba}: {len(df_depto)} registros.')

print('base_rh_por_departamento.xlsx gerada com abas por departamento!')

base_rh.xlsx (aba única) gerada!
Aba Todos:  todos os registros.
Aba Financeiro: 189 registros.
Aba Logística: 156 registros.
Aba Produção: 182 registros.
Aba RH: 166 registros.
Aba TI: 147 registros.
Aba Vendas: 160 registros.
base_rh_por_departamento.xlsx gerada com abas por departamento!
